# LangGraph Learning - Sentiment Analysis Pipeline

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.messages import add_messages
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict, Literal, Annotated

load_dotenv()

## State & Schema Definitions

In [ ]:
class SentimentAnalysis(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(description="the sentiment classification either positive or negative")
    confidence: float = Field(ge=0, le=1.0, description="confidence score from 0.0 to 1.0")
    reason: str = Field(description="brief explanation")

In [ ]:
class SentimentState(TypedDict):
    original_tweet: str
    sentiment: str
    confidence: float
    response_tweet: str

## LLM Setup

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

## Graph Nodes

In [ ]:
def analyze_sentiment(state: SentimentState):
    tweet = state["original_tweet"]
    messages = [
        SystemMessage("analyze sentiment and provide the structured output. use 0 to 1.0 scale for confidence. lower is negative and higher is positive"),
        HumanMessage(tweet)
    ]

    structured_llm = llm.with_structured_output(SentimentAnalysis)
    analysis = structured_llm.invoke(messages)
    return {
        'sentiment': analysis.sentiment,
        'confidence': analysis.confidence
    }

In [ ]:
def generate_positive_response(state: SentimentState):
    print(f"current state in positive response node: {state}")
    messages = [
        SystemMessage(f"Generate a warm response to this positive tweet under 280 chars. confidence: {state['confidence']}. High confidence means be enthusiastic otherwise be friendly."),
        HumanMessage(state['original_tweet'])
    ]

    response = llm.invoke(messages)
    return {'response_tweet': response.content.strip()}

In [ ]:
def generate_negative_response(state: SentimentState):
    print(f"current state in negative response node: {state}")
    messages = [
        SystemMessage(f"Generate an empathetic response to this negative tweet under 280 chars. confidence: {state['confidence']}."),
        HumanMessage(state['original_tweet'])
    ]

    response = llm.invoke(messages)
    return {'response_tweet': response.content.strip()}

In [ ]:
def route_sentiment(state: SentimentState):
    if state["sentiment"] == "positive":
        return "positive"
    else:
        return "negative"

## Build Graph

In [ ]:
builder = StateGraph(SentimentState)

builder.add_node("analyze_sentiment", analyze_sentiment)
builder.add_node("generate_positive_response", generate_positive_response)
builder.add_node("generate_negative_response", generate_negative_response)

builder.add_edge(START, "analyze_sentiment")
builder.add_conditional_edges(
    "analyze_sentiment",
    route_sentiment,
    {
        "positive": "generate_positive_response",
        "negative": "generate_negative_response"
    }
)
builder.add_edge("generate_positive_response", END)
builder.add_edge("generate_negative_response", END)

graph = builder.compile()

In [ ]:
graph

## Run

In [ ]:
result = graph.invoke({"original_tweet": "just launched my new product and loving the feedback!"})
print(result)

In [3]:
import time

from flask import Flask
app = Flask(__name__)

@app.route("/order")
def order():
    time.sleep(3)
    return {"order_id": "12345" ,"status":"confirmed"}

@app.route("/health")
def health():
    return {"status": "healthy"}

if __name__ == "__main__":
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

d:\vector_databse\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
